# Amazon Video Games Recommendation System

This project builds a personalized recommendation system on the [Amazon Reviews 2023 — Video Games](https://amazon-reviews-2023.github.io/data_processing/5core.html) dataset. It implements and compares two collaborative filtering approaches — **Matrix Factorization (MF)** and **Neural Collaborative Filtering (NCF)** — trained in PyTorch with time-based evaluation.

The pipeline covers data loading and sampling, preprocessing, a temporal train/test split, cold-start analysis, model training with early stopping, quantitative evaluation (RMSE, MAE), and a user-segment deep dive.

## Dataset

**Source:** [Amazon Reviews 2023 — Video Games (5-core)](https://amazon-reviews-2023.github.io/data_processing/5core.html)

The 5-core variant retains only users and items with at least five interactions, making it a clean, well-filtered starting point for collaborative filtering research. The full Video Games subset contains **814,586 interactions** across 94,762 unique users and 25,612 unique items, spanning October 1999 to September 2023.

Each record contains four fields: `user_id`, `parent_asin` (item identifier), `rating` (1–5 scale, explicit feedback), and `timestamp` (milliseconds since epoch).

> **To run this notebook:** download `Video_Games.csv.gz` from the source above and place it in the same directory as this notebook.

In [1]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Data file must be in the same directory as this notebook.
data_path = "Video_Games.csv.gz"

# Read the data into a pandas DataFrame.
df_raw = pd.read_csv(data_path, compression="gzip")

# Rename parent_asin to item_id so the data clearly matches the recommender format.
df = df_raw.rename(columns={"parent_asin": "item_id"}).copy()

# Convert timestamp from milliseconds since epoch to datetime.
df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="ms")

# The full dataset is large; a reproducible 200K-interaction sample (SEED=42) keeps
# training time manageable. Sampling precedes the temporal split so the train/test
# ordering by timestamp is preserved.
SAMPLE_SIZE = 200_000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

print("Dataset size after selecting the modeling subset:")
print(f"Number of interactions: {len(df):,}")

n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()
print(f"Number of unique users: {n_users:,}")
print(f"Number of unique items: {n_items:,}")


print("\nColumns:")
print(df.columns.tolist())

print("\nFirst few rows:")
display(df.head())

print("\nMissing values by column:")
display(df.isna().sum())

print("\nRating distribution:")
display(df["rating"].value_counts().sort_index())

dataset_source = "https://amazon-reviews-2023.github.io/data_processing/5core.html"
print("Dataset source:", dataset_source)

Dataset size after selecting the modeling subset:
Number of interactions: 200,000
Number of unique users: 80,521
Number of unique items: 23,588

Columns:
['user_id', 'item_id', 'rating', 'timestamp', 'timestamp_dt']

First few rows:


,user_id,item_id,rating,timestamp,timestamp_dt
0,AE5YOT3JJVXCOJSBZDLVVFVGBIFQ,B012JJTK9G,4.0,1461892711000,2016-04-29 01:18:31.000
1,AEARUPXRLM7YJDRUPPDO4ZGJDREA,B09VYJWTDN,5.0,1472491015000,2016-08-29 17:16:55.000
2,AFVI2AGXSXFRM7MS7N36ALNFBQ4Q,B07GLYDDRG,4.0,1538637250305,2018-10-04 07:14:10.305
3,AHFFDUOH2GUVJ3STQE3QU3B6F2EQ,B00FR7U15U,1.0,1400537555000,2014-05-19 22:12:35.000
4,AFL5WOPMIR4J4YSFUJIXYMUMZ6SQ,B00005N7YR,5.0,1397167254000,2014-04-10 22:00:54.000



Missing values by column:


user_id         0
item_id         0
rating          0
timestamp       0
timestamp_dt    0
dtype: int64


Rating distribution:


rating
1.0     12900
2.0      9117
3.0     17150
4.0     33070
5.0    127763
Name: count, dtype: int64

Dataset source: https://amazon-reviews-2023.github.io/data_processing/5core.html


## Modeling Strategy

The dataset provides explicit user ratings, making **preference-based collaborative filtering** the most natural approach. Two model families are implemented and compared:

- **Matrix Factorization (MF):** decomposes the user–item rating matrix into low-dimensional latent vectors, with per-user and per-item bias terms. It is a well-understood, computationally efficient baseline well-suited to sparse data.
- **Neural Collaborative Filtering (NCF):** replaces the dot-product interaction with a multi-layer perceptron, allowing the model to capture non-linear preference patterns — at the cost of requiring more data to generalize.

From a business perspective, these models enable personalized game discovery: they surface items a user is likely to rate highly based on the preferences of similar users, without requiring any item metadata. As interaction data grows, both models can be retrained to reflect evolving user tastes.

In [2]:
# Collaborative filtering is the most natural approach for this dataset:
# the key signal is historical user-item ratings, not item metadata.
# MF (dot-product) serves as the interpretable baseline; NCF (MLP) tests
# whether non-linear interactions add value at this scale.
print("Models to evaluate: Matrix Factorization (MF) vs Neural Collaborative Filtering (NCF)")

Models to evaluate: Matrix Factorization (MF) vs Neural Collaborative Filtering (NCF)


## Data Processing

The raw dataset is filtered to the five columns needed for collaborative filtering (`user_id`, `item_id`, `rating`, `timestamp`, `timestamp_dt`), null rows are dropped, ratings are cast to float, and duplicate user–item pairs are resolved by keeping the most recent interaction (sorted by `timestamp`).

In [3]:
# Keep only the columns needed for explicit-feedback collaborative filtering.
processed_data = df[["user_id", "item_id", "rating", "timestamp", "timestamp_dt"]].copy()

# Basic cleaning.
processed_data = processed_data.dropna(subset=["user_id", "item_id", "rating", "timestamp"]).copy()
processed_data["rating"] = processed_data["rating"].astype(float)

# Remove duplicate user-item pairs if any exist. This dataset should already have no duplicates,
# but this makes the workflow safer and easier to explain.
before_dedup = len(processed_data)
processed_data = (
    processed_data
    .sort_values("timestamp")
    .drop_duplicates(subset=["user_id", "item_id"], keep="last")
    .reset_index(drop=True)
)
after_dedup = len(processed_data)

print(f"Rows before duplicate removal: {before_dedup:,}")
print(f"Rows after duplicate removal: {after_dedup:,}")
print(f"Duplicates removed: {before_dedup - after_dedup:,}")

print("\nRating range:")
print(processed_data["rating"].min(), "to", processed_data["rating"].max())

print("\nTimestamp range:")
print(processed_data["timestamp_dt"].min(), "to", processed_data["timestamp_dt"].max())

print("\nProcessed data head:")
display(processed_data.head())

Rows before duplicate removal: 200,000
Rows after duplicate removal: 200,000
Duplicates removed: 0

Rating range:
1.0 to 5.0

Timestamp range:
1999-10-22 21:43:15 to 2023-08-29 06:50:54.782000

Processed data head:


,user_id,item_id,rating,timestamp,timestamp_dt
0,AHV2HBORH6QV3PYLTKF3CADFPOFA,B00000DMAM,5.0,940628595000,1999-10-22 21:43:15
1,AEVMJDGJSC664DBLKADJB7IBRFKA,B00000DMAD,5.0,943335580000,1999-11-23 05:39:40
2,AFKPKKYN6747GTCEC62R5FNJDZDA,B0000296ZH,3.0,943709017000,1999-11-27 13:23:37
3,AEVMJDGJSC664DBLKADJB7IBRFKA,B00000DMAN,5.0,943821874000,1999-11-28 20:44:34
4,AHJ2EGNNLVEMJA6MTCLFWWDY6QMQ,B000031KJX,3.0,944075741000,1999-12-01 19:15:41


## Train-Test Split

Interactions are sorted by timestamp. The earliest 80% of the sampled interactions form the training set; the latest 20% form the test set. This prevents any future information from leaking into the training window.

In [4]:
# Since timestamp is available, use a time-based split.
# The earliest 80% of interactions are used for training, and the latest 20% are used for testing.
processed_data = processed_data.sort_values("timestamp").reset_index(drop=True)
split_index = int(len(processed_data) * 0.80)

train_data = processed_data.iloc[:split_index].copy()
test_data = processed_data.iloc[split_index:].copy()

print("Sample training data:")
display(train_data.head())

print("\nSample test data:")
display(test_data.head())

print("\nDataset shapes:")
print(f"Training set shape: {train_data.shape}")
print(f"Test set shape: {test_data.shape}")

print("\nTime ranges:")
print("Training:", train_data["timestamp_dt"].min(), "to", train_data["timestamp_dt"].max())
print("Test:", test_data["timestamp_dt"].min(), "to", test_data["timestamp_dt"].max())

print("\nSplit percentage:")
print(f"Train percentage: {len(train_data) / len(processed_data):.2%}")
print(f"Test percentage: {len(test_data) / len(processed_data):.2%}")

Sample training data:


,user_id,item_id,rating,timestamp,timestamp_dt
0,AHV2HBORH6QV3PYLTKF3CADFPOFA,B00000DMAM,5.0,940628595000,1999-10-22 21:43:15
1,AEVMJDGJSC664DBLKADJB7IBRFKA,B00000DMAD,5.0,943335580000,1999-11-23 05:39:40
2,AFKPKKYN6747GTCEC62R5FNJDZDA,B0000296ZH,3.0,943709017000,1999-11-27 13:23:37
3,AEVMJDGJSC664DBLKADJB7IBRFKA,B00000DMAN,5.0,943821874000,1999-11-28 20:44:34
4,AHJ2EGNNLVEMJA6MTCLFWWDY6QMQ,B000031KJX,3.0,944075741000,1999-12-01 19:15:41



Sample test data:


,user_id,item_id,rating,timestamp,timestamp_dt
160000,AHIGIEENQJE7M7TEOS4GS4HF5KBA,B07DR59JLP,5.0,1575860778607,2019-12-09 03:06:18.607
160001,AF37I5XOOLCX5TMNVPFUUEN5ZA6A,B08XQLWT9V,5.0,1575864402296,2019-12-09 04:06:42.296
160002,AGPAKROYZGPYQ2NWQEMTONKN3YLQ,B089VKWRVP,4.0,1575864449011,2019-12-09 04:07:29.011
160003,AEJRUE5UU6IWJ6LLGNNCAMZLKGYA,B08NF1YWXD,5.0,1575865967098,2019-12-09 04:32:47.098
160004,AE6OPI4REOZIOXFXWYO4DLSDRIVQ,B001EYUQVE,5.0,1575871382611,2019-12-09 06:03:02.611



Dataset shapes:
Training set shape: (160000, 5)
Test set shape: (40000, 5)

Time ranges:
Training: 1999-10-22 21:43:15 to 2019-12-09 02:24:26.091000
Test: 2019-12-09 03:06:18.607000 to 2023-08-29 06:50:54.782000

Split percentage:
Train percentage: 80.00%
Test percentage: 20.00%


## Cold-Start Analysis

ID-embedding models cannot generate learned representations for users or items that never appeared during training. The following block identifies the fraction of test interactions that are affected by this constraint, and filters them out to create a **warm-start test set** for fair RMSE/MAE evaluation.

In [5]:
train_users = set(train_data["user_id"])
test_users = set(test_data["user_id"])
cold_start_users = test_users - train_users

print(f"Users in test but not in training: {len(cold_start_users):,}")

train_items = set(train_data["item_id"])
test_items = set(test_data["item_id"])
cold_start_items = test_items - train_items

print(f"Items in test but not in training: {len(cold_start_items):,}")

test_data["cold_start_user"] = ~test_data["user_id"].isin(train_users)
test_data["cold_start_item"] = ~test_data["item_id"].isin(train_items)
test_data["cold_start_instance"] = test_data["cold_start_user"] | test_data["cold_start_item"]

n_cold_instances = int(test_data["cold_start_instance"].sum())
print(f"Test instances with an unseen user or unseen item: {n_cold_instances:,}")
print(f"Cold-start instance percentage in test set: {n_cold_instances / len(test_data):.2%}")

# The models below are ID-embedding models, so they cannot generate learned embeddings for users/items
# that were never seen during training. For fair RMSE/MAE evaluation, keep a warm-start test set.
warm_test_data = test_data.loc[~test_data["cold_start_instance"]].copy()

print("\nWarm-start test set used for model evaluation:")
print(f"Warm-start test interactions: {len(warm_test_data):,}")
print(f"Warm-start test percentage: {len(warm_test_data) / len(test_data):.2%}")

cold_start_interpretation = f"""
The cold-start ratio is high in this dataset. This is not a coding error.
Amazon review data is sparse, and the time-based split means that many users or items appear only in the later test period.
Since Matrix Factorization and Neural Collaborative Filtering both rely on learned user and item embeddings, they cannot directly make reliable predictions for users or items that were never seen during training.

Therefore, I evaluate MF and NCF on the warm-start test subset, where both the user and item appeared in the training data.
In a real business setting, cold-start cases would need fallback strategies such as popular game recommendations, genre-based recommendations, item metadata, or onboarding questions to collect new user preferences.
"""
print(cold_start_interpretation)

Users in test but not in training: 12,707
Items in test but not in training: 5,418
Test instances with an unseen user or unseen item: 32,160
Cold-start instance percentage in test set: 80.40%

Warm-start test set used for model evaluation:
Warm-start test interactions: 7,840
Warm-start test percentage: 19.60%

The cold-start ratio is high in this dataset. This is not a coding error.
Amazon review data is sparse, and the time-based split means that many users or items appear only in the later test period.
Since Matrix Factorization and Neural Collaborative Filtering both rely on learned user and item embeddings, they cannot directly make reliable predictions for users or items that were never seen during training.

Therefore, I evaluate MF and NCF on the warm-start test subset, where both the user and item appeared in the training data.
In a real business setting, cold-start cases would need fallback strategies such as popular game recommendations, genre-based recommendations, item meta

## Models

Two collaborative filtering models are implemented:

1. **Matrix Factorization (MF)** — classic latent-factor model with user/item bias terms
2. **Neural Collaborative Filtering (NCF)** — MLP on concatenated user and item embeddings

Both use 32-dimensional embeddings, Adam optimization, MSE loss, and early stopping (patience = 3) evaluated on a held-out 10% validation split of the training data.

### Model Architecture

In [6]:
# Encode user_id and item_id based only on the training data.
user_to_idx = {user_id: idx for idx, user_id in enumerate(train_data["user_id"].unique())}
item_to_idx = {item_id: idx for idx, item_id in enumerate(train_data["item_id"].unique())}

num_users = len(user_to_idx)
num_items = len(item_to_idx)
global_mean_rating = train_data["rating"].mean()

print(f"Number of encoded training users: {num_users:,}")
print(f"Number of encoded training items: {num_items:,}")
print(f"Global mean rating in training set: {global_mean_rating:.4f}")

def add_encoded_ids(df):
    encoded = df.copy()
    encoded["user_idx"] = encoded["user_id"].map(user_to_idx)
    encoded["item_idx"] = encoded["item_id"].map(item_to_idx)
    return encoded.dropna(subset=["user_idx", "item_idx"]).copy()

train_encoded = add_encoded_ids(train_data)
test_encoded = add_encoded_ids(warm_test_data)

train_encoded["user_idx"] = train_encoded["user_idx"].astype("int64")
train_encoded["item_idx"] = train_encoded["item_idx"].astype("int64")
test_encoded["user_idx"] = test_encoded["user_idx"].astype("int64")
test_encoded["item_idx"] = test_encoded["item_idx"].astype("int64")

class RatingDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

from sklearn.model_selection import train_test_split

BATCH_SIZE = 4096

# Split the training data into a model-training set and a validation set.
# The validation set is used only for early stopping, not for final evaluation.
train_model_data, val_data = train_test_split(
    train_encoded,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

train_loader = DataLoader(RatingDataset(train_model_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(RatingDataset(val_data), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(RatingDataset(test_encoded), batch_size=BATCH_SIZE, shuffle=False)

print(f"Model-training interactions: {len(train_model_data):,}")
print(f"Validation interactions: {len(val_data):,}")
print(f"Warm-start test interactions: {len(test_encoded):,}")

class MatrixFactorization(nn.Module):
    """Classical matrix factorization with user/item embeddings and bias terms."""
    def __init__(self, num_users, num_items, embedding_dim=32, global_mean=0.0):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([global_mean], dtype=torch.float32))

        nn.init.normal_(self.user_embedding.weight, std=0.05)
        nn.init.normal_(self.item_embedding.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        interaction = (user_vec * item_vec).sum(dim=1)
        prediction = (
            self.global_bias
            + self.user_bias(user_idx).squeeze(-1)
            + self.item_bias(item_idx).squeeze(-1)
            + interaction
        )
        return prediction

class NeuralCollaborativeFiltering(nn.Module):
    """Neural collaborative filtering using user/item embeddings and an MLP."""
    def __init__(self, num_users, num_items, embedding_dim=32):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1)
        )

        nn.init.normal_(self.user_embedding.weight, std=0.05)
        nn.init.normal_(self.item_embedding.weight, std=0.05)

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        x = torch.cat([user_vec, item_vec], dim=1)
        return self.mlp(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mf_model = MatrixFactorization(num_users, num_items, embedding_dim=32, global_mean=global_mean_rating).to(device)
ncf_model = NeuralCollaborativeFiltering(num_users, num_items, embedding_dim=32).to(device)

print("\nModels created:")
print(mf_model)
print(ncf_model)

Number of encoded training users: 67,814
Number of encoded training items: 18,170
Global mean rating in training set: 4.2729
Model-training interactions: 144,000
Validation interactions: 16,000
Warm-start test interactions: 7,840
Device: cpu

Models created:
MatrixFactorization(
  (user_embedding): Embedding(67814, 32)
  (item_embedding): Embedding(18170, 32)
  (user_bias): Embedding(67814, 1)
  (item_bias): Embedding(18170, 1)
)
NeuralCollaborativeFiltering(
  (user_embedding): Embedding(67814, 32)
  (item_embedding): Embedding(18170, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


### Training

In [7]:
def evaluate_mse(model, data_loader):
    criterion = nn.MSELoss(reduction="sum")
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for users, items, ratings in data_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)

            predictions = model(users, items)
            total_loss += criterion(predictions, ratings).item()
            total_count += ratings.size(0)

    return total_loss / total_count

def train_model_with_early_stopping(
    model,
    train_loader,
    val_loader,
    epochs=20,
    learning_rate=0.001,
    weight_decay=1e-5,
    patience=3
):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_losses = []

        for users, items, ratings in train_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)

            optimizer.zero_grad()
            predictions = model(users, items)
            loss = criterion(predictions, ratings)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        avg_train_loss = float(np.mean(epoch_losses))
        avg_val_loss = evaluate_mse(model, val_loader)

        train_history.append(avg_train_loss)
        val_history.append(avg_val_loss)

        print(
            f"Epoch {epoch}: training MSE = {avg_train_loss:.4f}, "
            f"validation MSE = {avg_val_loss:.4f}"
        )

        # Early stopping: keep the model with the best validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered at epoch {epoch}. Best validation MSE = {best_val_loss:.4f}")
            break

    # Restore the best validation model before final test evaluation
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return train_history, val_history

EPOCHS = 20

print("Training Matrix Factorization model")
mf_history, mf_val_history = train_model_with_early_stopping(
    mf_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    learning_rate=0.005,
    weight_decay=1e-5,
    patience=3
)

print("\nTraining Neural Collaborative Filtering model")
ncf_history, ncf_val_history = train_model_with_early_stopping(
    ncf_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    learning_rate=0.001,
    weight_decay=1e-5,
    patience=3
)


Training Matrix Factorization model
Epoch 1: training MSE = 1.3581, validation MSE = 1.3273
Epoch 2: training MSE = 1.2202, validation MSE = 1.2930
Epoch 3: training MSE = 1.0076, validation MSE = 1.2726
Epoch 4: training MSE = 0.7079, validation MSE = 1.2703
Epoch 5: training MSE = 0.4390, validation MSE = 1.2807
Epoch 6: training MSE = 0.2671, validation MSE = 1.2851
Epoch 7: training MSE = 0.1726, validation MSE = 1.2854
Early stopping triggered at epoch 7. Best validation MSE = 1.2703

Training Neural Collaborative Filtering model
Epoch 1: training MSE = 17.4136, validation MSE = 12.9100
Epoch 2: training MSE = 5.3071, validation MSE = 1.7750
Epoch 3: training MSE = 1.3798, validation MSE = 1.4079
Epoch 4: training MSE = 0.9836, validation MSE = 1.4192
Epoch 5: training MSE = 0.8075, validation MSE = 1.4863
Epoch 6: training MSE = 0.7241, validation MSE = 1.5352
Early stopping triggered at epoch 6. Best validation MSE = 1.4079


### Evaluation

In [8]:
def predict(model, df, batch_size=4096):
    model.eval()
    dataset = RatingDataset(df)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    predictions = []
    actuals = []

    with torch.no_grad():
        for users, items, ratings in loader:
            users = users.to(device)
            items = items.to(device)
            preds = model(users, items).detach().cpu().numpy()
            predictions.extend(preds)
            actuals.extend(ratings.numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    # Ratings are on a 1-5 scale, so clamp predictions to the valid range before reporting metrics.
    predictions_clipped = np.clip(predictions, 1.0, 5.0)

    return predictions_clipped, actuals

mf_pred, actual = predict(mf_model, test_encoded)
ncf_pred, _ = predict(ncf_model, test_encoded)

prediction_examples = test_encoded[["user_id", "item_id", "rating"]].head(5).copy()
prediction_examples["MF_predicted_rating"] = mf_pred[:5]
prediction_examples["NCF_predicted_rating"] = ncf_pred[:5]

print("Five predicted values and their corresponding user-item pairs:")
display(prediction_examples)

def regression_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return rmse, mae

mf_rmse, mf_mae = regression_metrics(actual, mf_pred)
ncf_rmse, ncf_mae = regression_metrics(actual, ncf_pred)

evaluation_results = pd.DataFrame({
    "Model": ["Matrix Factorization", "Neural Collaborative Filtering"],
    "RMSE": [mf_rmse, ncf_rmse],
    "MAE": [mf_mae, ncf_mae]
})

print("\nOverall warm-start test-set performance:")
display(evaluation_results)

mf_best_epoch = int(np.argmin(mf_val_history))
mf_best_train_rmse = float(np.sqrt(mf_history[mf_best_epoch]))
mf_best_val_rmse = float(np.sqrt(mf_val_history[mf_best_epoch]))

interpretation = f"""
The task is numeric rating prediction, so RMSE and MAE are appropriate evaluation metrics.
Lower RMSE and MAE indicate better prediction accuracy.

Based on the warm-start test set, Matrix Factorization performs better than Neural Collaborative Filtering in this run, with a lower RMSE ({mf_rmse:.4f} vs {ncf_rmse:.4f}) and a lower MAE ({mf_mae:.4f} vs {ncf_mae:.4f}).
This means MF performs better both when larger prediction errors are penalized more heavily and when average absolute prediction error is considered.

For Matrix Factorization, the best validation model was selected at epoch {mf_best_epoch + 1}, with a training RMSE of approximately {mf_best_train_rmse:.4f} and a validation RMSE of approximately {mf_best_val_rmse:.4f}.
Its test RMSE is {mf_rmse:.4f}, which suggests that MF still has some generalization gap, although early stopping and weight decay helped reduce unnecessary memorization of the training data.

The weaker overall performance of Neural Collaborative Filtering may be because the neural model needs more interaction data per user and item to learn stable non-linear preference patterns.
With sparse rating data, the simpler Matrix Factorization structure can be more stable and easier to generalize.

From a business perspective, both models are most useful for users and products with enough historical interaction data.
Cold-start users and items, which represent a large share of the test set, would still require additional strategies such as popularity-based recommendations, onboarding preference surveys, genre-based recommendations, or item metadata.
"""
print(interpretation)

Five predicted values and their corresponding user-item pairs:


,user_id,item_id,rating,MF_predicted_rating,NCF_predicted_rating
160002,AGPAKROYZGPYQ2NWQEMTONKN3YLQ,B089VKWRVP,4.0,4.202377,4.194365
160008,AFEV76Y7FCUNGPIQLADFQNKHN2JA,B084QDTVC6,3.0,3.906966,4.307360
160010,AEDIKXKBO5WAHOHEQXNGFDYQRDYQ,B009VUHW60,5.0,4.545925,5.000000
160011,AFHT2WMBYDYEFSCAMQ3ABCIHII3Q,B00NOD0OTW,5.0,4.168575,3.915524
160013,AFR47TREMCF4NRH42AWI2ELGYDGA,B07MRJ12HX,5.0,4.308752,4.652595



Overall warm-start test-set performance:


,Model,RMSE,MAE
0,Matrix Factorization,1.308764,1.018809
1,Neural Collaborative Filtering,1.379688,1.058407



The task is numeric rating prediction, so RMSE and MAE are appropriate evaluation metrics.
Lower RMSE and MAE indicate better prediction accuracy.

Based on the warm-start test set, Matrix Factorization performs better than Neural Collaborative Filtering in this run, with a lower RMSE (1.3088 vs 1.3797) and a lower MAE (1.0188 vs 1.0584).
This means MF performs better both when larger prediction errors are penalized more heavily and when average absolute prediction error is considered.

For Matrix Factorization, the best validation model was selected at epoch 4, with a training RMSE of approximately 0.8414 and a validation RMSE of approximately 1.1271.
Its test RMSE is 1.3088, which suggests that MF still has some generalization gap, although early stopping and weight decay helped reduce unnecessary memorization of the training data.

The weaker overall performance of Neural Collaborative Filtering may be because the neural model needs more interaction data per user and item to learn 

### User-Segment Performance Analysis

To understand how well each model serves different types of users, warm-start test interactions are assigned to one of three equal-sized groups — **light**, **average**, and **power** users — based on each user's interaction count in the full training set. The tertile thresholds are computed on the distribution of training-interaction counts within the warm-start test population.

In [9]:
# Use the full training set (train_encoded) to count each user's historical interactions.
# train_encoded covers all training rows (before the 90/10 model-training/val split),
# giving a complete picture of how much history each user contributed.
user_interaction_counts = train_encoded.groupby("user_id").size().rename("train_user_interactions")

segmented_test = test_encoded.merge(
    user_interaction_counts,
    left_on="user_id",
    right_index=True,
    how="left"
)

# Warm-start test users are guaranteed to appear in train_encoded by construction;
# fillna(0) is a defensive fallback for any edge cases.
segmented_test["train_user_interactions"] = segmented_test["train_user_interactions"].fillna(0)

q33 = segmented_test["train_user_interactions"].quantile(1/3)
q67 = segmented_test["train_user_interactions"].quantile(2/3)

def user_segment(n):
    if n <= q33:
        return "Light users"
    elif n <= q67:
        return "Average users"
    else:
        return "Power users"

segmented_test["user_segment"] = segmented_test["train_user_interactions"].apply(user_segment)
segmented_test["mf_prediction"] = mf_pred
segmented_test["ncf_prediction"] = ncf_pred

segment_results = []

for segment_name, group in segmented_test.groupby("user_segment"):
    mf_segment_rmse, mf_segment_mae = regression_metrics(group["rating"].values, group["mf_prediction"].values)
    ncf_segment_rmse, ncf_segment_mae = regression_metrics(group["rating"].values, group["ncf_prediction"].values)

    segment_results.append({
        "User Segment": segment_name,
        "Number of Test Interactions": len(group),
        "Average Train Interactions per User": group["train_user_interactions"].mean(),
        "MF_RMSE": mf_segment_rmse,
        "MF_MAE": mf_segment_mae,
        "NCF_RMSE": ncf_segment_rmse,
        "NCF_MAE": ncf_segment_mae
    })

segment_results = pd.DataFrame(segment_results).sort_values("Average Train Interactions per User")

print("Segmentation criteria:")
print(f"Light users: training interactions <= {q33:.1f}")
print(f"Average users: training interactions > {q33:.1f} and <= {q67:.1f}")
print(f"Power users: training interactions > {q67:.1f}")

print("\nSegment-level model performance:")
display(segment_results)

segment_interpretation = """
The deep dive shows that model performance differs by user history. Light users are harder to predict because they have fewer interactions in the training data, so the models have less behavioral information to learn from.
MF performs more consistently across light, average, and power users, with lower RMSE in all three segments.
For MAE, MF performs better for light and average users, while NCF performs better for power users.
This suggests that the simpler latent-factor structure of MF is more stable overall, but the more flexible NCF model may produce smaller typical absolute errors when users have richer interaction histories.
One useful takeaway from this segment analysis is that one recommendation strategy may not work equally well for all users.
For low-history or new users, the platform should use simpler or fallback recommendations, such as popular games, genre-based recommendations, or onboarding preference collection.
For power users, more expressive models like NCF may become useful as the platform collects more interaction history, although MF remains the more stable model in this experiment.
"""
print(segment_interpretation)

Segmentation criteria:
Light users: training interactions <= 1.0
Average users: training interactions > 1.0 and <= 2.0
Power users: training interactions > 2.0

Segment-level model performance:


,User Segment,Number of Test Interactions,Average Train Interactions per User,MF_RMSE,MF_MAE,NCF_RMSE,NCF_MAE
1,Light users,3740,1.000000,1.320573,1.034964,1.406474,1.128056
0,Average users,1869,2.000000,1.309144,1.016490,1.365323,1.029730
2,Power users,2231,5.613178,1.288400,0.993669,1.345907,0.965673



The deep dive shows that model performance differs by user history. Light users are harder to predict because they have fewer interactions in the training data, so the models have less behavioral information to learn from.
MF performs more consistently across light, average, and power users, with lower RMSE in all three segments.
For MAE, MF performs better for light and average users, while NCF performs better for power users.
This suggests that the simpler latent-factor structure of MF is more stable overall, but the more flexible NCF model may produce smaller typical absolute errors when users have richer interaction histories.
One useful takeaway from this segment analysis is that one recommendation strategy may not work equally well for all users.
For low-history or new users, the platform should use simpler or fallback recommendations, such as popular games, genre-based recommendations, or onboarding preference collection.
For power users, more expressive models like NCF may bec

## Limitations and Future Directions

**Current limitations:**
- **Cold-start:** ~80% of test interactions are cold-start cases (users or items first seen in the test period). MF and NCF cannot make learned predictions for these without additional signals such as item metadata or user onboarding surveys.
- **Sparsity:** the dataset is highly sparse. Most users have very few ratings, limiting what the models can learn about individual preferences.
- **Explicit feedback only:** ratings reflect post-purchase opinions; they do not capture browsing, search, or wishlist behavior that might be more informative about interest.

**Potential future directions:**
- **Cold-start mitigation:** integrate item-level features (genre, developer, platform) via Wide & Deep or a content-based hybrid layer.
- **Sequential signals:** Amazon timestamp data could power SASRec or similar sequence-aware models that capture evolving preferences.
- **Implicit feedback:** incorporate view/click signals alongside ratings for a fuller picture of user intent.
- **Periodic retraining:** a production deployment would retrain on a rolling window to keep embeddings fresh as new users and games appear.